In [2]:
from google.colab import files

uploaded = files.upload()

Saving sample_text.txt to sample_text.txt


In [5]:
import string
from collections import defaultdict


# ==============================
# Trie Data Structure
# ==============================

class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_end = True

    def _dfs(self, node, prefix, results):
        if node.is_end:
            results.append(prefix)
        for char in node.children:
            self._dfs(node.children[char], prefix + char, results)

    def autocomplete(self, prefix):
        node = self.root
        for char in prefix:
            if char not in node.children:
                return []
            node = node.children[char]

        results = []
        self._dfs(node, prefix, results)
        return results


# ==============================
# Smart Text Analyzer
# ==============================

class SmartTextAnalyzer:

    def __init__(self):
        self.raw_text = ""
        self.cleaned_text = ""
        self.words = []
        self.sentences = []
        self.word_counts = {}
        self.char_counts = {}
        self.unique_words = set()
        self.bigrams = defaultdict(lambda: defaultdict(int))
        self.trigrams = defaultdict(lambda: defaultdict(int))
        self.history = []  # Stack for Undo
        self.trie = Trie()

        self.stop_words = {
            "the", "is", "and", "in", "of", "to", "a", "an",
            "on", "for", "with", "that", "this", "it"
        }

        self.positive_words = {"good", "great", "excellent", "amazing", "love", "happy"}
        self.negative_words = {"bad", "terrible", "awful", "hate", "sad", "poor"}

    # -------------------------
    # Loading Text
    # -------------------------

    def load_text(self):
        choice = input("Load from file (f) or direct input (d)? [f/d]: ").lower()

        if choice == "f":
            path = input("Enter file path: ")
            try:
                with open(path, "r", encoding="utf-8") as file:
                    self.raw_text = file.read()
                print("Text loaded successfully!")
            except FileNotFoundError:
                print("File not found.")
                return False
        else:
            print("Enter text. Type $$END_TEXT$$ on new line to finish:")
            lines = []
            while True:
                line = input()
                if line == "$$END_TEXT$$":
                    break
                lines.append(line)
            self.raw_text = "\n".join(lines)
            print("Text entered successfully!")

        return True

    # -------------------------
    # Preprocessing
    # -------------------------

    def preprocess(self):
        text = self.raw_text.lower()

        for char in string.punctuation:
            text = text.replace(char, "")

        self.cleaned_text = text
        self.words = text.split()
        self.sentences = self.raw_text.split(".")
        self.unique_words = set(self.words)

        self.build_word_counts()
        self.build_char_counts()
        self.build_ngrams()
        self.build_trie()

        print("Preprocessing completed successfully!")

    def build_word_counts(self):
        self.word_counts = {}
        for word in self.words:
            self.word_counts[word] = self.word_counts.get(word, 0) + 1

    def build_char_counts(self):
        self.char_counts = {}
        for char in self.cleaned_text:
            if char not in [" ", "\n", "\t"]:
                self.char_counts[char] = self.char_counts.get(char, 0) + 1

    def build_ngrams(self):
        self.bigrams = defaultdict(lambda: defaultdict(int))
        self.trigrams = defaultdict(lambda: defaultdict(int))

        for i in range(len(self.words) - 1):
            self.bigrams[self.words[i]][self.words[i + 1]] += 1

        for i in range(len(self.words) - 2):
            pair = self.words[i] + " " + self.words[i + 1]
            self.trigrams[pair][self.words[i + 2]] += 1

    def build_trie(self):
        self.trie = Trie()
        for word in self.unique_words:
            self.trie.insert(word)

    # -------------------------
    # Core Features
    # -------------------------

    def word_statistics(self):
        print("\nTotal Words:", len(self.words))
        print("Unique Words:", len(self.unique_words))
        print("Top 5 Words:")
        sorted_words = sorted(self.word_counts.items(),
                              key=lambda x: x[1], reverse=True)
        for word, count in sorted_words[:5]:
            print(word, ":", count)

    def character_statistics(self):
        total_chars = sum(self.char_counts.values())
        print("\nTotal Characters (no spaces):", total_chars)
        for char, count in sorted(self.char_counts.items()):
            print(char, ":", count)

    def search(self):
        query = input("Enter word to search: ").lower()
        found = False

        for i, sentence in enumerate(self.sentences):
            clean_sentence = sentence.lower()
            words_in_sentence = clean_sentence.split()

            for index, word in enumerate(words_in_sentence):
                if query == word:
                    print(f"Found in sentence {i+1}, word position {index+1}")
                    print("Sentence:", sentence.strip())
                    found = True

        if not found:
            print("Not found.")

    def replace_word(self):
        old = input("Word to replace: ").lower()
        new = input("New word: ").lower()

        self.history.append(self.words.copy())

        self.words = [new if w == old else w for w in self.words]
        self.raw_text = " ".join(self.words)

        self.preprocess()
        print("Replacement done successfully!")

    def undo(self):
        if not self.history:
            print("Nothing to undo.")
            return

        self.words = self.history.pop()
        self.raw_text = " ".join(self.words)

        self.preprocess()
        print("Undo successful!")

    # -------------------------
    # Intelligent Features
    # -------------------------

    def predict_next_word(self):
        word = input("Enter a word: ").lower()
        if word in self.bigrams:
            predictions = sorted(self.bigrams[word].items(),
                                 key=lambda x: x[1], reverse=True)
            for w, c in predictions[:5]:
                print(w, ":", c)
        else:
            print("No prediction available.")

    def predict_trigram(self):
        phrase = input("Enter two words: ").lower()
        if phrase in self.trigrams:
            predictions = sorted(self.trigrams[phrase].items(),
                                 key=lambda x: x[1], reverse=True)
            for w, c in predictions[:5]:
                print(w, ":", c)
        else:
            print("No trigram prediction available.")

    def autocomplete(self):
        prefix = input("Enter prefix: ").lower()
        suggestions = self.trie.autocomplete(prefix)
        print("Suggestions:", suggestions[:10])

    def sentiment_analysis(self):
        sentence = input("Enter sentence: ").lower()
        words = sentence.split()
        pos = 0
        neg = 0

        i = 0
        while i < len(words):
            if words[i] == "not" and i + 1 < len(words):
                if words[i+1] in self.positive_words:
                    neg += 1
                    i += 2
                    continue
                elif words[i+1] in self.negative_words:
                    pos += 1
                    i += 2
                    continue

            if words[i] in self.positive_words:
                pos += 1
            elif words[i] in self.negative_words:
                neg += 1
            i += 1

        if pos > neg:
            print("Sentiment: Positive 😊")
        elif neg > pos:
            print("Sentiment: Negative 😞")
        else:
            print("Sentiment: Neutral 😐")

    def extract_keywords(self):
        filtered = {w: c for w, c in self.word_counts.items()
                    if w not in self.stop_words}

        sorted_words = sorted(filtered.items(),
                              key=lambda x: x[1], reverse=True)

        print("Top Keywords:")
        for word, count in sorted_words[:10]:
            print(word, ":", count)

    # Spell Check

    def edit_distance(self, w1, w2):
        m, n = len(w1), len(w2)
        dp = [[0]*(n+1) for _ in range(m+1)]

        for i in range(m+1):
            dp[i][0] = i
        for j in range(n+1):
            dp[0][j] = j

        for i in range(1, m+1):
            for j in range(1, n+1):
                if w1[i-1] == w2[j-1]:
                    dp[i][j] = dp[i-1][j-1]
                else:
                    dp[i][j] = 1 + min(
                        dp[i-1][j],
                        dp[i][j-1],
                        dp[i-1][j-1]
                    )

        return dp[m][n]

    def spell_check(self):
        word = input("Enter word to check: ").lower()

        if word in self.word_counts:
            print("Word is correct.")
            return

        suggestions = []

        for vocab_word in self.unique_words:
            distance = self.edit_distance(word, vocab_word)
            if distance <= 2:
                suggestions.append((vocab_word, distance))

        suggestions.sort(key=lambda x: x[1])

        if suggestions:
            print("Did you mean:")
            for w, d in suggestions[:5]:
                print(f"{w} (distance={d})")
        else:
            print("No suggestions found.")

    # Word Cloud Data

    def generate_wordcloud_data(self):
        max_freq = max(self.word_counts.values())
        normalized = []

        for word, freq in self.word_counts.items():
            normalized.append((word, round(freq / max_freq, 3)))

        normalized.sort(key=lambda x: x[1], reverse=True)

        print("\nWord Cloud Data (Normalized):")
        for word, value in normalized[:20]:
            print(word, ":", value)

    # -------------------------
    # Menu
    # -------------------------

    def menu(self):
        while True:
            print("\n===== Smart Text Analyzer =====")
            print("1. Word Statistics")
            print("2. Character Statistics")
            print("3. Search Word")
            print("4. Replace Word")
            print("5. Undo")
            print("6. Predict Next Word (Bigram)")
            print("7. Predict Next Word (Trigram)")
            print("8. Smart Autocomplete")
            print("9. Sentiment Analysis")
            print("10. Extract Keywords")
            print("11. Spell Check")
            print("12. Generate Word Cloud Data")
            print("13. Exit")

            choice = input("Choose option: ")

            if choice == "1":
                self.word_statistics()
            elif choice == "2":
                self.character_statistics()
            elif choice == "3":
                self.search()
            elif choice == "4":
                self.replace_word()
            elif choice == "5":
                self.undo()
            elif choice == "6":
                self.predict_next_word()
            elif choice == "7":
                self.predict_trigram()
            elif choice == "8":
                self.autocomplete()
            elif choice == "9":
                self.sentiment_analysis()
            elif choice == "10":
                self.extract_keywords()
            elif choice == "11":
                self.spell_check()
            elif choice == "12":
                self.generate_wordcloud_data()
            elif choice == "13":
                print("Goodbye 👋")
                break
            else:
                print("Invalid choice.")


# ==============================
# Run Program
# ==============================

if __name__ == "__main__":
    analyzer = SmartTextAnalyzer()
    if analyzer.load_text():
        analyzer.preprocess()
        analyzer.menu()

Load from file (f) or direct input (d)? [f/d]: f
Enter file path: sample_text.txt
Text loaded successfully!
Preprocessing completed successfully!

===== Smart Text Analyzer =====
1. Word Statistics
2. Character Statistics
3. Search Word
4. Replace Word
5. Undo
6. Predict Next Word (Bigram)
7. Predict Next Word (Trigram)
8. Smart Autocomplete
9. Sentiment Analysis
10. Extract Keywords
11. Spell Check
12. Generate Word Cloud Data
13. Exit
Choose option: 1

Total Words: 127
Unique Words: 86
Top 5 Words:
is : 7
and : 6
data : 5
science : 4
learning : 4

===== Smart Text Analyzer =====
1. Word Statistics
2. Character Statistics
3. Search Word
4. Replace Word
5. Undo
6. Predict Next Word (Bigram)
7. Predict Next Word (Trigram)
8. Smart Autocomplete
9. Sentiment Analysis
10. Extract Keywords
11. Spell Check
12. Generate Word Cloud Data
13. Exit
Choose option: 2

Total Characters (no spaces): 735
a : 61
b : 9
c : 31
d : 29
e : 88
f : 11
g : 30
h : 13
i : 67
j : 2
k : 3
l : 35
m : 33
n : 56
o :